In [12]:
import numpy as np
import sys
import os
from autokmc.structure import build_surface, build_nanoparticle
from ase.visualize.x3d import view_x3d
from autokmc.surface import find_surface_atoms
import copy
from autokmc.graph import build_graph

In [ ]:
# Ensure the project root is on the path when running from the autokmc/ subdirectory
sys.path.insert(0, os.path.abspath("../.."))

In [ ]:
### Load the Allegro/NequIP calculator
from nequip.ase import NequIPCalculator

_MODEL_PATH = os.path.join(os.path.dirname(__file__) if "__file__" in dir() else ".", "asehcocuau.nequip.pt2")

def make_calc():
    """Return a fresh NequIPCalculator instance loaded from the .pt2 model."""
    return NequIPCalculator.from_compiled_model(
        compile_path=_MODEL_PATH,
        device="cuda",
        #species_to_type_name={"H": "H", "C": "C", "O": "O", "Cu": "Cu", "Au": "Au"},
    )

calc = make_calc()
print(f"Calculator : {calc.__class__.__name__}")
print(f"Model      : {_MODEL_PATH}")

In [ ]:
## Build a Cu(111) surface slab
slab = build_surface(
    composition="Cu",
    crystal_structure="fcc",
    miller_index=(1, 1, 1),
    calculator=make_calc(),
    min_slab_size=8.0,
    min_vacuum_size=12.0,
    goal_x=12.0,
    goal_y=12.0,
    n_freeze_layers=2,
    verbose=True,
    orthogonalise=True
)

print(f"\nSlab formula : {slab.get_chemical_formula()}")
print(f"Slab atoms   : {len(slab)}")
cell = slab.get_cell()
print(f"Cell (Å)     : a={cell[0,0]:.3f}  b={cell[1,1]:.3f}  c={cell[2,2]:.3f}")

In [ ]:
## Visualise the slab
view_x3d(slab)

In [ ]:
### Get surface atoms

surface_mask, surface_indices, method = find_surface_atoms(
    slab,
    which="top",
    tag_atoms=True,   # writes slab.arrays["surface"] for extxyz export
)

print(f"Detection method : {method}")
print(f"Surface atoms    : {surface_mask.sum()} / {len(slab)}")
print(f"Surface indices  : {surface_indices}")

In [ ]:
### Visualise surface atoms in X3D
# Surface atoms are shown as Au (gold), bulk atoms remain as Cu
# so the two populations are visually distinct in x3d.
slab_vis = copy.deepcopy(slab)
symbols = np.array(slab_vis.get_chemical_symbols())
symbols[surface_indices] = "Au"
slab_vis.set_chemical_symbols(symbols.tolist())

view_x3d(slab_vis)

In [ ]:
### Build the graph for the slab
graph = build_graph(slab)
#print number of bonds (edges) and number of nodes (atoms)
print(f"Graph has {graph.number_of_nodes()} nodes and {graph.number_of_edges()} edges.")
# Show breakdown by node type
from collections import Counter
type_counts = Counter(d["type"] for _, d in graph.nodes(data=True))
for t, n in sorted(type_counts.items()):
    print(f"  {t:10s} : {n}")

In [ ]:
### Build reactant graphs for CO and O2
from autokmc.reactants import build_reactant

co = build_reactant("[C-]#[O+]", calculator=make_calc())
o2 = build_reactant("O=O",       calculator=make_calc())
c  = build_reactant("[C]",       calculator=make_calc())

for r in [co, o2, c]:
    print(f"\nReactant : {r.smiles}")
    print(f"  Formula  : {r.atoms.get_chemical_formula()}")
    print(f"  Atoms    : {len(r.atoms)}")
    print(f"  Nodes    : {r.graph.number_of_nodes()}")
    print(f"  Edges    : {r.graph.number_of_edges()}")
    for i, d in r.graph.nodes(data=True):
        pos = d['position']
        print(f"    node {i}  element={d['element']:2s}  type={d['type']}  r_cov={d['covalent_radius']:.3f} Å  pos=({pos[0]:.3f}, {pos[1]:.3f}, {pos[2]:.3f}) Å")

In [ ]:
### Find adsorption sites — C (single atom)
from autokmc.site import find_adsorption_sites

sites_c, site_graph_c = find_adsorption_sites(
    graph, c,
    calculator=make_calc(),
    slab=slab,
    n_freeze_layers=2,
    verbose=True,
)

print(f"\nC sites summary")
from collections import Counter
print(f"  Total sites  : {len(sites_c)}")
for iso_cid, count in sorted(Counter(s.iso_class for s in sites_c).items()):
    rep = next(s for s in sites_c if s.iso_class == iso_cid)
    e_str = f"  E_ads={rep.energy:.4f} eV  converged={rep.converged}" if rep.energy is not None else ""
    print(f"  iso-class {iso_cid:2d}  {rep.site_type:8s}  {count:3d} sites{e_str}  "
          f"pos example = ({rep.position[0]:.3f}, {rep.position[1]:.3f}, {rep.position[2]:.3f}) Å")

In [ ]:
### Visualise C sites — Plotly top-view (matching workflow_surface.py §5.10)
import plotly.graph_objects as go
import plotly.colors as pc_colors

surface_mask_c, surface_indices_c, _ = (
    surface_mask, surface_indices, None
)

cell_arr = slab.get_cell()
cell_x   = float(cell_arr[0, 0])
cell_y   = float(cell_arr[1, 1])
pos_all  = slab.get_positions()

iso_cids_c  = sorted({s.iso_class for s in sites_c})
palette     = pc_colors.qualitative.Alphabet
cid_color_c = {c: palette[k % len(palette)] for k, c in enumerate(iso_cids_c)}

_site_label_c = lambda n: {1: "top", 2: "bridge", 3: "hollow"}.get(n, f"{n}-fold")

vis = []

# Surface atoms (background)
vis.append(go.Scatter(
    x=pos_all[surface_indices, 0], y=pos_all[surface_indices, 1],
    mode="markers",
    marker=dict(size=18, color="#e8794a", opacity=0.30, line=dict(width=0)),
    name="surface atoms", hoverinfo="skip",
))

# Edges between adjacent sites
ex_list, ey_list = [], []
for na, nb in site_graph_c.edges():
    pa = site_graph_c.nodes[na]["position"]
    pb = site_graph_c.nodes[nb]["position"]
    delta = pb[:2] - pa[:2]
    delta[0] -= np.round(delta[0] / cell_x) * cell_x
    delta[1] -= np.round(delta[1] / cell_y) * cell_y
    pb_uw = pa[:2] + delta
    ex_list += [pa[0], pb_uw[0], None]
    ey_list += [pa[1], pb_uw[1], None]
vis.append(go.Scatter(
    x=ex_list, y=ey_list, mode="lines",
    line=dict(color="#aaaaaa", width=0.8),
    showlegend=False, hoverinfo="skip",
))

# Site nodes coloured by iso-class
for cid in iso_cids_c:
    nodes_c = [n for n, d in site_graph_c.nodes(data=True) if d["iso_class"] == cid]
    pts_c   = np.array([site_graph_c.nodes[n]["position"] for n in nodes_c])
    label   = _site_label_c(site_graph_c.nodes[nodes_c[0]]["n_conn"])
    # Show E_ads in hover if available
    e_list  = [next((s.energy for s in sites_c if s.iso_class == cid and s.energy is not None), None)]
    e_str   = f"  E_ads={e_list[0]:.4f} eV" if e_list[0] is not None else ""
    vis.append(go.Scatter(
        x=pts_c[:, 0], y=pts_c[:, 1],
        mode="markers",
        marker=dict(size=8, color=cid_color_c[cid], symbol="diamond", opacity=0.92,
                    line=dict(width=0.5, color="#333333")),
        name=f"iso-class {cid}  ({label}){e_str}  [{len(nodes_c)} sites]",
        hovertemplate=(f"<b>{label}</b>  iso-class {cid}{e_str}<br>"
                       "x=%{x:.2f}  y=%{y:.2f}<extra></extra>"),
    ))

fig_sg = go.Figure(data=vis)
fig_sg.update_layout(
    title=dict(text=f"Cu(111) – C adsorption-site graph  |  "
                    f"{site_graph_c.number_of_nodes()} sites  |  "
                    f"{site_graph_c.number_of_edges()} adjacencies",
               font_size=13),
    xaxis=dict(title="x (Å)", scaleanchor="y", scaleratio=1, range=[-0.5, cell_x + 0.5]),
    yaxis=dict(title="y (Å)", range=[-0.5, cell_y + 0.5]),
    legend=dict(x=1.01, y=0.99, font_size=10),
    margin=dict(l=10, r=220, t=55, b=10),
    width=900, height=700,
    plot_bgcolor="#f7f7f7",
)
fig_sg.show()

In [ ]:
### Find adsorption sites — CO (multi-atom)
sites_co, site_graph_co = find_adsorption_sites(
    graph, co,
    calculator=make_calc(),
    slab=slab,
    n_freeze_layers=2,
    n_orientations=200,
    verbose=True,
)

print(f"\nCO sites summary")
print(f"  Total sites  : {len(sites_co)}")
for iso_cid, count in sorted(Counter(s.iso_class for s in sites_co).items()):
    rep = next(s for s in sites_co if s.iso_class == iso_cid)
    n_surf = len({sg for _, sg in rep.conn_global})
    print(f"  iso-class {iso_cid:2d}  {rep.site_type:8s}  {count:3d} sites  "
          f"{n_surf} surface atoms bonded")